In [3]:
import pandas as pd # 读数据 + 洗数据
import numpy as np # 算数据
import matplotlib.pyplot as plt # 画图工具
from sklearn.preprocessing import MinMaxScaler #归一化工具
from sklearn.model_selection import train_test_split # 划分训练集和测试集
from tensorflow import keras # 神经网络工具
from tensorflow.keras import layers # 全连接层
from sklearn.metrics import classification_report # 分类报告工具

# 解决画图可能导致的中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

### 数据处理

In [4]:
# 1. 导入数据集
dataset = pd.read_csv("air_quality_synthetic.csv")

# 2. 分离特征 (X) 和 标签 (Y)
# 假设最后一列是你要预测的值（比如空气污染指数），前面所有列是特征
X = dataset.iloc[:, :-1].values  # 取所有行，除了最后一列之外的所有列
Y = dataset.iloc[:, -1].values   # 取所有行，只取最后一列

# 3. 先划分：把数据切成训练集和测试集 (8:2 划分)
# random_state=42 保证每次切分结果一致
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 4. 后归一化（极其关键的纪律！）
scaler_X = MinMaxScaler()

# 4.1 对训练集特征：既计算 min/max，又进行缩放 (fit_transform)
X_train_scaled = scaler_X.fit_transform(X_train)

# 4.2 对测试集特征：只用训练集的 min/max 进行缩放 (transform)
X_test_scaled = scaler_X.transform(X_test)

# ==========================================
# 进阶：如果你的 Y（标签）也是连续数值，做回归任务也需要归一化
# ==========================================
# 注意：Y 通常是一维的，MinMaxScaler 要求输入二维，所以需要 reshape(-1, 1)
scaler_Y = MinMaxScaler()

Y_train_scaled = scaler_Y.fit_transform(Y_train.reshape(-1, 1))
Y_test_scaled = scaler_Y.transform(Y_test.reshape(-1, 1))

### 模型搭建

In [8]:
# 初始化模型
model = keras.Sequential()

# 1. 先单独加一个 Input 层，专门负责接收数据
model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

# 2. 第一个隐藏层
model.add(layers.Dense(10, activation='relu'))

# 3. 第二个隐藏层
model.add(layers.Dense(10, activation='relu'))

# 4. 输出层：回归任务，1个神经元，绝对不加 activation（保持纯线性）
model.add(layers.Dense(1))

# 5. 打印模型结构
model.summary()

# 编译
model.compile(
    optimizer='adam',        # 优化器：推荐用 Adam，收敛快且稳
    loss='mse',              # 损失函数：均方误差（回归任务的标配）
    metrics=['mae']          # 评估指标：平均绝对误差（人类更容易看懂的误差）
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 10)             │            70 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 191 (764.00 B)

 Trainable params: 191 (764.00 B)

 Non-trainable params: 0 (0.00 B)